In [3]:
# Εγκατάσταση των βιβλιοθηκών για Fine-Tuning
# Εγκατάσταση ΧΩΡΙΣ το flash-attn
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Ρύθμιση του Quantization (για να χωρέσει το μοντέλο στην GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Επιλογή του Base Model (Το Llama-3-8B είναι η κορυφαία επιλογή)
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Σημείωση: Θα χρειαστείς ένα Hugging Face Token για να κατεβάσεις το Llama.
# Εναλλακτικά χρησιμοποιούμε το "mistralai/Mistral-7B-v0.1" που είναι ανοιχτό.

In [5]:
from datasets import load_dataset

# 1. Φόρτωση με streaming=True για να αποφύγουμε το ReadTimeout
dataset_stream = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train", streaming=True)

# Παίρνουμε ένα μικρό δείγμα (π.χ. 1000 παραδείγματα) για να ξεκινήσουμε γρήγορα
# Το streaming δεν υποστηρίζει slicing στυλ [:10%], οπότε χρησιμοποιούμε take()
dataset_sample = dataset_stream.take(1000)

def formatting_prompts_func(example):
    # Στο streaming mode, η συνάρτηση παίρνει ΕΝΑ παράδειγμα τη φορά
    instruction = example["query"]
    output      = example["answer"]
    return {"text": f"<s>[INST] {instruction} [/INST] {output} </s>"}

# Εφαρμογή της μορφοποίησης
# Στο streaming mode το map λειτουργεί λίγο διαφορετικά
dataset = dataset_sample.map(formatting_prompts_func)

# Έλεγχος: Εκτύπωση του πρώτου στοιχείου
for item in dataset:
    print("Το format είναι έτοιμο:")
    print(item['text'][:500] + "...") # Τυπώνουμε μόνο την αρχή για να μη γεμίσει η οθόνη
    break

Το format είναι έτοιμο:
<s>[INST] Create a nested loop to print every combination of numbers between 0-9, excluding any combination that contains the number 5. Additionally, exclude any combination that contains a repeating digit. Implement the solution without using any built-in functions or libraries to check for repeating digits. [/INST] Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:

```python...


In [1]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Μετατρέπουμε το δείγμα σε λίστα (γιατί κάναμε restart)
train_data = list(dataset)

# 2. Ρυθμίσεις εκπαίδευσης
training_args = TrainingArguments(
    output_dir="./ai-code-reviewer-results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,  # 100 βήματα είναι αρκετά για να δούμε αν το Loss πέφτει
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none"
)

# 3. Αρχικοποίηση
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

print("Ξεκινάμε! Αν δεις το progress bar, το 'θηρίο' άρχισε να μαθαίνει...")
trainer.train()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/memory.py)

In [2]:
# 1. Αναβάθμιση όλων των κρίσιμων βιβλιοθηκών στις τελευταίες εκδόσεις
!pip install -U -q transformers accelerate peft trl datasets bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.4 MB/s eta 0:00:00


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset

# 1. Φόρτωση Μοντέλου & Tokenizer (αν χάθηκαν λόγω restart)
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Απαραίτητο για το padding

# 2. Ρύθμιση LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 3. Φόρτωση Dataset (Streaming)
dataset_stream = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train", streaming=True)
dataset_sample = dataset_stream.take(500) # Λιγότερα δείγματα για σιγουριά στην αρχή

def format_func(ex):
    return {"text": f"<s>[INST] {ex['query']} [/INST] {ex['answer']} </s>"}

train_data = list(dataset_sample.map(format_func))

# 4. Παράμετροι Εκπαίδευσης
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=50, # Ξεκινάμε με 50 για να δούμε ότι δουλεύει
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim="paged_adamw_32bit",
    report_to="none"
)

# 5. Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=256, # Μικρότερο μήκος για αποφυγή OOM
    args=training_args,
)

print("Πάμε άλλη μια! Η εκπαίδευση ξεκινά...")
trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1. Προετοιμασία των Tokens (Tokenization)
# Πρέπει να μετατρέψουμε το κείμενο σε νούμερα πριν το στείλουμε στον Trainer
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

# Μετατρέπουμε τη λίστα μας σε Dataset αντικείμενο της Hugging Face
from datasets import Dataset
hf_dataset = Dataset.from_list(train_data)
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# 2. Ορισμός κλασικών Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=50,
    fp16=True,
    logging_steps=5,
    optim="paged_adamw_32bit",
    report_to="none",
    remove_unused_columns=True # Σημαντικό για να μην κρασάρει με extra πεδία
)

# 3. Χρήση του βασικού Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Παράκαμψη SFTTrainer επιτυχής. Ξεκινάμε το training...")
trainer.train()

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Παράκαμψη SFTTrainer επιτυχής. Ξεκινάμε το training...


Step,Training Loss
5,1.285572
10,1.128265
15,0.987881
20,0.882223
25,0.867447
30,0.854918
35,0.884636
40,0.775131
45,0.885108
50,0.861448


TrainOutput(global_step=50, training_loss=0.9412630271911621, metrics={'train_runtime': 243.061, 'train_samples_per_second': 0.823, 'train_steps_per_second': 0.206, 'total_flos': 2186488578048000.0, 'train_loss': 0.9412630271911621, 'epoch': 0.4})

In [6]:
# 1. Θέτουμε το μοντέλο σε evaluation mode
model.eval()

# 2. Φτιάχνουμε ένα prompt με ένα bug (π.χ. ένα κλασικό Index Error ή Typo)
test_prompt = "<s>[INST] Find the bug in this Python code: \ndef calculate_average(numbers):\n    total = sum(numbrs)\n    return total / len(numbers) [/INST]"

# 3. Tokenization και παραγωγή απάντησης
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2, # Χαμηλό για πιο "σοβαρές" απαντήσεις
        do_sample=True
    )

print("Απάντηση του AI Code Reviewer:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Απάντηση του AI Code Reviewer:
[INST] Find the bug in this Python code: 
def calculate_average(numbers):
    total = sum(numbrs)
    return total / len(numbers) [/INST] The bug in the code is that the variable `numbrs` is misspelled as `numbrs` instead of `numbers` in the `sum` function. The correct code should be:

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)


In [8]:
# Αποθήκευση των εκπαιδευμένων adapters
model.save_pretrained("./ai_code_reviewer_adapter")
tokenizer.save_pretrained("./ai_code_reviewer_adapter")

print("Οι adapters αποθηκεύτηκαν στον φάκελο 'ai_code_reviewer_adapter'")

Οι adapters αποθηκεύτηκαν στον φάκελο 'ai_code_reviewer_adapter'


In [9]:
import shutil
from google.colab import files

# Συμπίεση του φακέλου σε .zip
shutil.make_archive("ai_code_reviewer_adapter", 'zip', "ai_code_reviewer_adapter")

# Λήψη του αρχείου
files.download("ai_code_reviewer_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>